# Autoresearch Experiment Analysis

Analysis of autonomous trading strategy optimization results from `results.tsv`.

**Setup**: `pip install pandas matplotlib`

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv("results.tsv", sep="\t")
df["idx"] = range(len(df))
print(f"Loaded {len(df)} experiments")
df.head()

In [ ]:
# Experiment outcomes
counts = df["status"].value_counts()
print("Experiment outcomes:")
print(counts.to_string())
print(f"\nKeep rate: {counts.get('keep', 0) / len(df) * 100:.1f}%")

## Score Trajectory

Track how avg_score evolves as experiments progress. Green = kept, Red = discarded.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

# Filter out crashes for plotting
valid = df[df["status"] != "crash"].copy()
colors = ["#00FFA3" if s == "keep" else "#FF3366" for s in valid["status"]]

ax.scatter(valid["idx"], valid["avg_score"], c=colors, s=40, alpha=0.7, edgecolors="white", linewidth=0.5)

# Connect kept experiments
kept = valid[valid["status"] == "keep"]
if len(kept) > 1:
    ax.plot(kept["idx"], kept["avg_score"], color="#00FFA3", alpha=0.4, linewidth=1)

ax.set_xlabel("Experiment #", fontsize=12)
ax.set_ylabel("avg_score", fontsize=12)
ax.set_title("Autoresearch: Trading Strategy Score Trajectory", fontsize=14)
ax.set_facecolor("#09090B")
fig.set_facecolor("#09090B")
ax.tick_params(colors="white")
ax.xaxis.label.set_color("white")
ax.yaxis.label.set_color("white")
ax.title.set_color("white")
for spine in ax.spines.values():
    spine.set_color("#333")
ax.grid(True, alpha=0.15, color="white")

plt.tight_layout()
plt.show()

## Best Score Progression

Running maximum of kept experiments — the optimization frontier.

In [ ]:
kept = df[df["status"] == "keep"].copy()
if len(kept) > 0:
    kept["best_so_far"] = kept["avg_score"].cummax()

    fig, ax = plt.subplots(figsize=(16, 5))
    ax.step(kept["idx"], kept["best_so_far"], color="#00FFA3", linewidth=2.5, where="post")
    ax.fill_between(kept["idx"], kept["best_so_far"], alpha=0.1, color="#00FFA3", step="post")
    ax.scatter(kept["idx"], kept["avg_score"], color="#00E5FF", s=25, alpha=0.6, zorder=5)

    ax.set_xlabel("Experiment #", fontsize=12)
    ax.set_ylabel("Best avg_score", fontsize=12)
    ax.set_title("Best Score Over Time (Monotonic Frontier)", fontsize=14)
    ax.set_facecolor("#09090B")
    fig.set_facecolor("#09090B")
    ax.tick_params(colors="white")
    ax.xaxis.label.set_color("white")
    ax.yaxis.label.set_color("white")
    ax.title.set_color("white")
    for spine in ax.spines.values():
        spine.set_color("#333")
    ax.grid(True, alpha=0.15, color="white")

    plt.tight_layout()
    plt.show()
else:
    print("No kept experiments yet.")

## Summary Statistics

In [ ]:
kept = df[df["status"] == "keep"].copy()
baseline_score = df.iloc[0]["avg_score"] if len(df) > 0 else 0

print(f"Total experiments:    {len(df)}")
print(f"Kept:                {len(df[df['status'] == 'keep'])}")
print(f"Discarded:           {len(df[df['status'] == 'discard'])}")
print(f"Crashed:             {len(df[df['status'] == 'crash'])}")
print(f"Keep rate:           {len(df[df['status'] == 'keep']) / max(len(df), 1) * 100:.1f}%")
print(f"")
print(f"Baseline score:      {baseline_score}")
print(f"Best score:          {df['avg_score'].max():.2f}")
print(f"Improvement:         +{df['avg_score'].max() - baseline_score:.2f} ({(df['avg_score'].max() - baseline_score) / max(abs(baseline_score), 0.01) * 100:.1f}%)")
print(f"")
if len(kept) > 0:
    best_row = kept.loc[kept["avg_score"].idxmax()]
    print(f"Best experiment:     {best_row['description']}")
    print(f"Best commit:         {best_row['commit']}")

## Top Improvements (Kept Experiments by Score)

In [ ]:
kept = df[df["status"] == "keep"].copy()
if len(kept) > 1:
    kept["delta"] = kept["avg_score"].diff()
    top = kept.dropna(subset=["delta"]).nlargest(10, "delta")
    print("Top 10 improvements:\n")
    for _, row in top.iterrows():
        print(f"  +{row['delta']:.2f}  →  {row['avg_score']:.2f}  |  {row['description']}")
else:
    print("Not enough kept experiments for delta analysis.")